## Import


In [1]:

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
 
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import precision_recall_curve
 
print("=" * 60)
print("ISOLATION FOREST — Unsupervised Fraud Anomaly Detection")
print("=" * 60)
 
mlflow.set_experiment("FraudDetection_IsolationForest")


StatementMeta(, 6afaba6c-f64b-4788-a5e9-3ad2dde3d922, 3, Finished, Available, Finished, False)

ISOLATION FOREST — Unsupervised Fraud Anomaly Detection


2026/05/07 13:45:43 INFO mlflow.tracking.fluent: Experiment with name 'FraudDetection_IsolationForest' does not exist. Creating a new experiment.


<Experiment: artifact_location='sds://onelakecentralindia.pbidedicated.windows.net/c9a67f7c-cad6-4156-8afd-be7ae3dea097/1d7360a7-fedf-42f2-b7d5-8c49e4ae777e', creation_time=1778161544589, experiment_id='1d7360a7-fedf-42f2-b7d5-8c49e4ae777e', last_update_time=1778161544589, lifecycle_stage='active', name='FraudDetection_IsolationForest', tags={}>

##### What is Isolation Forest? 
###### How Isolation Forest works:
---------------------------
1. Randomly select a feature from the dataset
2. Randomly select a split value between min and max of that feature
3. Repeat until each data point is isolated in its own leaf
 
Note: FRAUD transactions are isolated in FEWER splits
because they are statistical rare and unusual values.
LEGITIMATE transactions require MORE splits because they cluster
with many similar normal transactions.
 
The anomaly score = average path length across many trees.
Short path -> anomaly,  Long path -> normal.
 
Why use this for fraud:
  - Does NOT need labelled fraud data to train
  - Catches new fraud patterns the supervised model hasn't seen
  - Works even with 0.17% positive rate
  - Computationally efficient at 284K rows

## Load Features

In [2]:
FEAT_COLS = [f"V{i}" for i in range(1, 29)] + ["Amount", "hour_of_day"]
 
df_pd = spark.read.format("delta").table("silver_features") \
             .select(["Class"] + FEAT_COLS) \
             .toPandas()
 
X = df_pd[FEAT_COLS].fillna(0)
y = df_pd["Class"]
 
print(f"Total transactions : {len(X):,}")
print(f"Fraud transactions : {y.sum():,} ({y.mean():.4%})")

StatementMeta(, 6afaba6c-f64b-4788-a5e9-3ad2dde3d922, 4, Finished, Available, Finished, False)

Total transactions : 284,807
Fraud transactions : 492 (0.1727%)


## Split for Evaluation

In [3]:
# Train Isolation Forest on ONLY legitimate transactions.
# This is the correct approach: teach the model what
# "normal" looks like; anomalies become detectable by contrast.
# If we trained on all data including fraud, the model would
# learn to treat fraud patterns as "normal".
X_legit  = X[y == 0]
X_fraud  = X[y == 1]
print(f"\nTraining on {len(X_legit):,} legitimate transactions")
print(f"Will evaluate against {len(X_fraud):,} fraud transactions")

StatementMeta(, 6afaba6c-f64b-4788-a5e9-3ad2dde3d922, 5, Finished, Available, Finished, False)


Training on 284,315 legitimate transactions
Will evaluate against 492 fraud transactions


##  Contamination Parameter 

In [4]:
# contamination = expected fraction of anomalies in the dataset
# Our fraud rate is 0.17% — but Isolation Forest is used
# in a production stream where fraud rate is even lower.
# Set slightly higher (0.002) to err on the side of caution.

CONTAMINATION = 0.002  # 0.2% — slightly above true 0.17%
 
print(f"\nContamination parameter: {CONTAMINATION}")
print(f"  True fraud rate: {y.mean():.4%}")
print(f"  Setting higher to bias toward flagging anomalies")

StatementMeta(, 6afaba6c-f64b-4788-a5e9-3ad2dde3d922, 6, Finished, Available, Finished, False)


Contamination parameter: 0.002
  True fraud rate: 0.1727%
  Setting higher to bias toward flagging anomalies


##  Train Isolation Forest

In [5]:
with mlflow.start_run(run_name="IsolationForest_v1"):
 
    iso = IsolationForest(
        n_estimators  = 200,
        contamination = CONTAMINATION,
        max_samples   = "auto",   # min(256, n_samples)
        max_features  = 1.0,      # use all features
        bootstrap     = False,    # sampling without replacement
        random_state  = 42,
        n_jobs        = -1,
        verbose       = 0,
    )
 
    iso.fit(X_legit)
    print("✅ Isolation Forest trained")
 
    # Score ALL transactions 
    # decision_function: more negative = more anomalous
    # predict: -1 = anomaly (fraud), 1 = normal (legitimate)
    raw_scores  = iso.decision_function(X)
    predictions = iso.predict(X)
 
    # Normalise to 0–1: lower decision score -> higher fraud probability
    fraud_scores = 1 - (
        (raw_scores - raw_scores.min()) /
        (raw_scores.max() - raw_scores.min())
    )
 
    #  Metrics 
    auc = roc_auc_score(y, fraud_scores)
    ap  = average_precision_score(y, fraud_scores)
 
    # Find optimal threshold via precision-recall curve
    precision, recall, thresholds = precision_recall_curve(y, fraud_scores)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)
    best_idx  = np.argmax(f1_scores)
    best_thr  = thresholds[best_idx]
    best_prec = precision[best_idx]
    best_rec  = recall[best_idx]
 
    flagged = (predictions == -1).sum()
    actual_caught = y[predictions == -1].sum()
 
    # Log to MLflow
    mlflow.log_params({
        "n_estimators":  200,
        "contamination": CONTAMINATION,
        "max_samples":   "auto",
        "bootstrap":     False,
    })
    mlflow.log_metrics({
        "AUC_ROC":         round(auc, 4),
        "Avg_Precision":   round(ap, 4),
        "best_F1":         round(f1_scores[best_idx], 4),
        "best_precision":  round(best_prec, 4),
        "best_recall":     round(best_rec, 4),
        "flagged_count":   int(flagged),
        "fraud_caught":    int(actual_caught),
    })
    mlflow.sklearn.log_model(
        iso, "isolation_forest",
        registered_model_name="FraudIsolationForest"
    )
 
    print(f"\n{'='*50}")
    print("ISOLATION FOREST RESULTS")
    print("="*50)
    print(f"AUC-ROC           : {auc:.4f}")
    print(f"Avg Precision     : {ap:.4f}")
    print(f"Best F1 score     : {f1_scores[best_idx]:.4f}")
    print(f"  at threshold    : {best_thr:.4f}")
    print(f"  precision       : {best_prec:.4f}")
    print(f"  recall          : {best_rec:.4f}")
    print(f"\nFlagged anomalies : {flagged:,} ({flagged/len(X)*100:.2f}%)")
    print(f"Actual fraud caught: {actual_caught:,} of {y.sum():,} ({actual_caught/y.sum():.0%})")
    print(f"False positives   : {flagged-actual_caught:,}")

StatementMeta(, 6afaba6c-f64b-4788-a5e9-3ad2dde3d922, 7, Finished, Available, Finished, False)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
Successfully registered model 'FraudIsolationForest'.


✅ Isolation Forest trained

ISOLATION FOREST RESULTS
AUC-ROC           : 0.9478
Avg Precision     : 0.1141
Best F1 score     : 0.2343
  at threshold    : 0.7060
  precision       : 0.2204
  recall          : 0.2500

Flagged anomalies : 702 (0.25%)
Actual fraud caught: 133 of 492 (27%)
False positives   : 569
